In [132]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName('Final_Assignment').getOrCreate()

# 1. Basic DataFrame Operations

* Load the sales.csv and customer.csv files into separate DataFrames.
* Display the schema of both DataFrames.
* Show the first 5 rows from the sales DataFrame.
* Count the number of rows and columns in the customer DataFrame.

In [133]:
# Load the sales.csv and customer.csv files into separate DataFrames.
sales_df = spark.read.option('InferSchema', True)\
                     .option('Header', True)\
                     .csv('sales.txt')

customer_df = spark.read.option('InferSchema', True)\
                     .option('Header', True)\
                     .csv('customer.txt')


In [134]:
# Display the schema of both DataFrames.
print('Schema of Sales Dataframe :- ')
sales_df.printSchema()

print('Schema of Customer Dataframe :- ')
customer_df.printSchema()

Schema of Sales Dataframe :- 
root
 |-- sales_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- region: string (nullable = true)

Schema of Customer Dataframe :- 
root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)



In [135]:
# Show the first 5 rows from the sales DataFrame.
print('first five rows of Sales Dataframe :- ')
sales_df.show(5)

first five rows of Sales Dataframe :- 
+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+-----------+-------+------+----------+------+
|       1|        101| Laptop| 50000|2023-01-15| North|
|       2|        102| Mobile| 15000|2023-02-10| South|
|       3|        103| Tablet| 20000|2023-03-05|  West|
|       4|        104| Laptop| 55000|2023-03-15|  East|
|       5|        105|Desktop| 40000|2023-04-20| North|
+--------+-----------+-------+------+----------+------+
only showing top 5 rows


In [136]:
# Count the number of rows and columns in the customer DataFrame.
print(f'\nnumber of rows in customer Dataframe :- {customer_df.count()} and number of columns in the customer DataFrame :- {len(customer_df.columns )}')


number of rows in customer Dataframe :- 7 and number of columns in the customer DataFrame :- 5


# 2. Data Cleaning
*  Remove duplicate rows from the sales DataFrame based on customer_id,product,amount,sale_date,region columns
*  Drop rows where any column in the customer DataFrame has null values.
*  Replace null values in the amount column of the sales DataFrame with 0.
*  Replace null values in the email column of the customer DataFrame with the value
"unknown".

In [137]:
# Remove duplicate rows from the sales DataFrame based on customer_id,product,amount,sale_date,region columns
print('rows before dropping duplicates in sales_df :- ',sales_df.count())
sales_df = sales_df.dropDuplicates(subset=['customer_id','product','amount','sale_date','region'])
print('rows after dropping duplicates in sales_df :- ',sales_df.count())


rows before dropping duplicates in sales_df :-  10
rows after dropping duplicates in sales_df :-  10


In [138]:
# Drop rows where any column in the customer DataFrame has null values.
print('rows before dropping duplicates in customer_df :- ',customer_df.count())
customer_df = customer_df.na.drop(how='any')
print('rows after dropping duplicates in customer_df :- ',customer_df.count())

rows before dropping duplicates in customer_df :-  7
rows after dropping duplicates in customer_df :-  7


In [139]:
# Replace null values in the amount column of the sales DataFrame with 0.
sales_df = sales_df.na.fill({'amount': 0})

In [140]:
# Replace null values in the email column of the customer DataFrame with the value "unknown".
customer_df = customer_df.na.fill({'email' : 'unknown'})

# 3. Column Manipulation
* Add a new column discounted_amount to the sales DataFrame that applies a 10%
discount on amount.
* Rename the city column in the customer DataFrame to customer_city.
* Drop the region column from the sales DataFrame.
* Create a new column customer_age_category in the customer DataFrame based on age:
   * "Youth" for age < 30
   * "Adult" for 30 <= age < 50
   * "Senior" for age >= 50

In [141]:
# Add a new column discounted_amount to the sales DataFrame that applies a 10% discount on amount.
sales_df = sales_df.withColumn('discounted_amount', round(col('amount') - col('amount') * 0.1, 2))
print('sales_df after applying 10% discount on amount :- ')
sales_df.show()

sales_df after applying 10% discount on amount :- 
+--------+-----------+-------+------+----------+------+-----------------+
|sales_id|customer_id|product|amount| sale_date|region|discounted_amount|
+--------+-----------+-------+------+----------+------+-----------------+
|       1|        101| Laptop| 50000|2023-01-15| North|          45000.0|
|       3|        103| Tablet| 20000|2023-03-05|  West|          18000.0|
|       7|        102| Laptop| 60000|2023-06-15|  East|          54000.0|
|      10|        105| Laptop| 70000|2023-09-25| North|          63000.0|
|       5|        105|Desktop| 40000|2023-04-20| North|          36000.0|
|       2|        102| Mobile| 15000|2023-02-10| South|          13500.0|
|       9|        104|Desktop| 45000|2023-08-10|  West|          40500.0|
|       6|        101| Mobile| 15000|2023-05-10| South|          13500.0|
|       8|        103| Tablet| 20000|2023-07-05| North|          18000.0|
|       4|        104| Laptop| 55000|2023-03-15|  East|      

In [142]:
# Rename the city column in the customer DataFrame to customer_city.
customer_df = customer_df.withColumnRenamed('city' ,'customer_city')
print('customer_df after renaming city column to customer_city column :- ')
customer_df.show()

customer_df after renaming city column to customer_city column :- 
+-----------+-------------+--------------------+---+-------------+
|customer_id|customer_name|               email|age|customer_city|
+-----------+-------------+--------------------+---+-------------+
|        101|  Arun Sharma|arun.sharma@email...| 28|        Delhi|
|        102|  Meena Verma|meena.verma@email...| 34|       Mumbai|
|        103|  Rahul Yadav|rahul.yadav@email...| 30|    Bangalore|
|        104|  Priya Patel|priya.patel@email...| 27|    Ahmedabad|
|        105|  Sneha Reddy|sneha.reddy@email...| 29|    Hyderabad|
|        106|   Vikas Jain|vikas.jain@email.com| 31|      Chennai|
|        107|     Amit Roy|  amit.roy@email.com| 35|      Kolkata|
+-----------+-------------+--------------------+---+-------------+



In [143]:
# Drop the region column from the sales DataFrame.
sales_df_copy = sales_df.drop('region')
print('sales_df after dropping region column :- ')
sales_df_copy.show()

sales_df after dropping region column :- 
+--------+-----------+-------+------+----------+-----------------+
|sales_id|customer_id|product|amount| sale_date|discounted_amount|
+--------+-----------+-------+------+----------+-----------------+
|       1|        101| Laptop| 50000|2023-01-15|          45000.0|
|       3|        103| Tablet| 20000|2023-03-05|          18000.0|
|       7|        102| Laptop| 60000|2023-06-15|          54000.0|
|      10|        105| Laptop| 70000|2023-09-25|          63000.0|
|       5|        105|Desktop| 40000|2023-04-20|          36000.0|
|       2|        102| Mobile| 15000|2023-02-10|          13500.0|
|       9|        104|Desktop| 45000|2023-08-10|          40500.0|
|       6|        101| Mobile| 15000|2023-05-10|          13500.0|
|       8|        103| Tablet| 20000|2023-07-05|          18000.0|
|       4|        104| Laptop| 55000|2023-03-15|          49500.0|
+--------+-----------+-------+------+----------+-----------------+



In [144]:
'''
Create a new column customer_age_category in the customer DataFrame based on age:
"Youth" for age < 30
"Adult" for 30 <= age < 50
"Senior" for age >= 50
'''

customer_df = customer_df.withColumn('customer_age_category',
                         when(col('age') < 30, 'Youth')
                         .when((col('age') >= 30) & (col('age') < 50), 'Adult')
                         .otherwise('Senior')
)
print('customer_df after adding customer_age_category column :- ')
customer_df.show()

customer_df after adding customer_age_category column :- 
+-----------+-------------+--------------------+---+-------------+---------------------+
|customer_id|customer_name|               email|age|customer_city|customer_age_category|
+-----------+-------------+--------------------+---+-------------+---------------------+
|        101|  Arun Sharma|arun.sharma@email...| 28|        Delhi|                Youth|
|        102|  Meena Verma|meena.verma@email...| 34|       Mumbai|                Adult|
|        103|  Rahul Yadav|rahul.yadav@email...| 30|    Bangalore|                Adult|
|        104|  Priya Patel|priya.patel@email...| 27|    Ahmedabad|                Youth|
|        105|  Sneha Reddy|sneha.reddy@email...| 29|    Hyderabad|                Youth|
|        106|   Vikas Jain|vikas.jain@email.com| 31|      Chennai|                Adult|
|        107|     Amit Roy|  amit.roy@email.com| 35|      Kolkata|                Adult|
+-----------+-------------+--------------------+---+

# 4. Filtering
* Filter the sales DataFrame to show only rows where amount is greater than 50,000.
* Filter the customer DataFrame to show customers aged between 25 and 30.
* Identify all customers who have made purchases in more than one region.
* Filter the top 3 sales based on amount for each product.

In [145]:
# Filter the sales DataFrame to show only rows where amount is greater than 50,000.
sales_df.filter('amount > 50000').show()

+--------+-----------+-------+------+----------+------+-----------------+
|sales_id|customer_id|product|amount| sale_date|region|discounted_amount|
+--------+-----------+-------+------+----------+------+-----------------+
|       7|        102| Laptop| 60000|2023-06-15|  East|          54000.0|
|      10|        105| Laptop| 70000|2023-09-25| North|          63000.0|
|       4|        104| Laptop| 55000|2023-03-15|  East|          49500.0|
+--------+-----------+-------+------+----------+------+-----------------+



In [146]:
# Filter the customer DataFrame to show customers aged between 25 and 30.
customer_df.filter('age >= 25 and age <= 30').show()

+-----------+-------------+--------------------+---+-------------+---------------------+
|customer_id|customer_name|               email|age|customer_city|customer_age_category|
+-----------+-------------+--------------------+---+-------------+---------------------+
|        101|  Arun Sharma|arun.sharma@email...| 28|        Delhi|                Youth|
|        103|  Rahul Yadav|rahul.yadav@email...| 30|    Bangalore|                Adult|
|        104|  Priya Patel|priya.patel@email...| 27|    Ahmedabad|                Youth|
|        105|  Sneha Reddy|sneha.reddy@email...| 29|    Hyderabad|                Youth|
+-----------+-------------+--------------------+---+-------------+---------------------+



In [147]:
# Identify all customers who have made purchases in more than one region.
sales_df.groupBy('customer_id')\
        .agg(count_distinct('region').alias('count_regions'))\
        .where(col('count_regions') > 1)\
        .show()

+-----------+-------------+
|customer_id|count_regions|
+-----------+-------------+
|        101|            2|
|        103|            2|
|        102|            2|
|        104|            2|
+-----------+-------------+



In [148]:
# Filter the top 3 sales based on amount for each product.
from pyspark.sql.window import Window

window_spec = Window.partitionBy('product').orderBy(col('amount').desc())

sales_df.withColumn('row_number', row_number().over(window_spec))\
        .filter('row_number <= 3')\
        .show()

+--------+-----------+-------+------+----------+------+-----------------+----------+
|sales_id|customer_id|product|amount| sale_date|region|discounted_amount|row_number|
+--------+-----------+-------+------+----------+------+-----------------+----------+
|       9|        104|Desktop| 45000|2023-08-10|  West|          40500.0|         1|
|       5|        105|Desktop| 40000|2023-04-20| North|          36000.0|         2|
|      10|        105| Laptop| 70000|2023-09-25| North|          63000.0|         1|
|       7|        102| Laptop| 60000|2023-06-15|  East|          54000.0|         2|
|       4|        104| Laptop| 55000|2023-03-15|  East|          49500.0|         3|
|       2|        102| Mobile| 15000|2023-02-10| South|          13500.0|         1|
|       6|        101| Mobile| 15000|2023-05-10| South|          13500.0|         2|
|       3|        103| Tablet| 20000|2023-03-05|  West|          18000.0|         1|
|       8|        103| Tablet| 20000|2023-07-05| North|          

# 5. Joins
* Perform an inner join between sales and customer DataFrames on customer_id.
* Perform a left join to include all records from sales and matching records from
customer.
* Perform a full outer join between sales and customer DataFrames.
* Identify customers who have not made any purchases by performing an anti-join.

In [149]:
# Perform an inner join between sales and customer DataFrames on customer_id.
sales_df.join(customer_df, on = 'customer_id', how = 'inner').show()

+-----------+--------+-------+------+----------+------+-----------------+-------------+--------------------+---+-------------+---------------------+
|customer_id|sales_id|product|amount| sale_date|region|discounted_amount|customer_name|               email|age|customer_city|customer_age_category|
+-----------+--------+-------+------+----------+------+-----------------+-------------+--------------------+---+-------------+---------------------+
|        101|       1| Laptop| 50000|2023-01-15| North|          45000.0|  Arun Sharma|arun.sharma@email...| 28|        Delhi|                Youth|
|        103|       3| Tablet| 20000|2023-03-05|  West|          18000.0|  Rahul Yadav|rahul.yadav@email...| 30|    Bangalore|                Adult|
|        102|       7| Laptop| 60000|2023-06-15|  East|          54000.0|  Meena Verma|meena.verma@email...| 34|       Mumbai|                Adult|
|        105|      10| Laptop| 70000|2023-09-25| North|          63000.0|  Sneha Reddy|sneha.reddy@email..

In [150]:
# Perform a left join to include all records from sales and matching records from customer.
sales_df.join(customer_df, on = 'customer_id', how = 'left').show()

+-----------+--------+-------+------+----------+------+-----------------+-------------+--------------------+---+-------------+---------------------+
|customer_id|sales_id|product|amount| sale_date|region|discounted_amount|customer_name|               email|age|customer_city|customer_age_category|
+-----------+--------+-------+------+----------+------+-----------------+-------------+--------------------+---+-------------+---------------------+
|        101|       1| Laptop| 50000|2023-01-15| North|          45000.0|  Arun Sharma|arun.sharma@email...| 28|        Delhi|                Youth|
|        103|       3| Tablet| 20000|2023-03-05|  West|          18000.0|  Rahul Yadav|rahul.yadav@email...| 30|    Bangalore|                Adult|
|        102|       7| Laptop| 60000|2023-06-15|  East|          54000.0|  Meena Verma|meena.verma@email...| 34|       Mumbai|                Adult|
|        105|      10| Laptop| 70000|2023-09-25| North|          63000.0|  Sneha Reddy|sneha.reddy@email..

In [151]:
# Perform a full outer join between sales and customer DataFrames.
sales_df.join(customer_df, on = 'customer_id', how = 'full_outer').show()

+-----------+--------+-------+------+----------+------+-----------------+-------------+--------------------+---+-------------+---------------------+
|customer_id|sales_id|product|amount| sale_date|region|discounted_amount|customer_name|               email|age|customer_city|customer_age_category|
+-----------+--------+-------+------+----------+------+-----------------+-------------+--------------------+---+-------------+---------------------+
|        101|       1| Laptop| 50000|2023-01-15| North|          45000.0|  Arun Sharma|arun.sharma@email...| 28|        Delhi|                Youth|
|        101|       6| Mobile| 15000|2023-05-10| South|          13500.0|  Arun Sharma|arun.sharma@email...| 28|        Delhi|                Youth|
|        102|       7| Laptop| 60000|2023-06-15|  East|          54000.0|  Meena Verma|meena.verma@email...| 34|       Mumbai|                Adult|
|        102|       2| Mobile| 15000|2023-02-10| South|          13500.0|  Meena Verma|meena.verma@email..

In [152]:
# Identify customers who have not made any purchases by performing an anti-join.
customer_df.join(sales_df, on = 'customer_id', how = 'anti').show()

+-----------+-------------+--------------------+---+-------------+---------------------+
|customer_id|customer_name|               email|age|customer_city|customer_age_category|
+-----------+-------------+--------------------+---+-------------+---------------------+
|        106|   Vikas Jain|vikas.jain@email.com| 31|      Chennai|                Adult|
|        107|     Amit Roy|  amit.roy@email.com| 35|      Kolkata|                Adult|
+-----------+-------------+--------------------+---+-------------+---------------------+



# 6. Aggregations
* Calculate the total sales amount for each product.
* Find the average age of customers in the customer DataFrame.
* Calculate the maximum and minimum sales amounts in the sales DataFrame.
* Group the customer DataFrame by customer_city and count the number of customers in
each city.

In [153]:
# Calculate the total sales amount for each product.
sales_df.groupBy('product').agg(sum('amount').alias('total_sales_amount')).show()

+-------+------------------+
|product|total_sales_amount|
+-------+------------------+
| Laptop|            235000|
| Mobile|             30000|
| Tablet|             40000|
|Desktop|             85000|
+-------+------------------+



In [154]:
# Find the average age of customers in the customer DataFrame.
customer_df.select(round(avg('age'), scale = 2).alias('average_age')).show()

+-----------+
|average_age|
+-----------+
|      30.57|
+-----------+



In [155]:
# Calculate the maximum and minimum sales amounts in the sales DataFrame.
sales_df.select(max('amount').alias('maximum_sale_amount'), min('amount').alias('minimum_sale_amount')).show()

+-------------------+-------------------+
|maximum_sale_amount|minimum_sale_amount|
+-------------------+-------------------+
|              70000|              15000|
+-------------------+-------------------+



In [156]:
# Group the customer DataFrame by customer_city and count the number of customers in each city.
customer_df.groupBy('customer_city').agg(count('customer_id').alias('count_customer_city')).show()

+-------------+-------------------+
|customer_city|count_customer_city|
+-------------+-------------------+
|    Bangalore|                  1|
|      Chennai|                  1|
|       Mumbai|                  1|
|    Ahmedabad|                  1|
|      Kolkata|                  1|
|        Delhi|                  1|
|    Hyderabad|                  1|
+-------------+-------------------+



# 7. Sorting
* Sort the sales DataFrame by amount in descending order.
* Sort the customer DataFrame by age in ascending order.

In [157]:
# Sort the sales DataFrame by amount in descending order.
sales_df.sort(col('amount').desc()).show()

+--------+-----------+-------+------+----------+------+-----------------+
|sales_id|customer_id|product|amount| sale_date|region|discounted_amount|
+--------+-----------+-------+------+----------+------+-----------------+
|      10|        105| Laptop| 70000|2023-09-25| North|          63000.0|
|       7|        102| Laptop| 60000|2023-06-15|  East|          54000.0|
|       4|        104| Laptop| 55000|2023-03-15|  East|          49500.0|
|       1|        101| Laptop| 50000|2023-01-15| North|          45000.0|
|       9|        104|Desktop| 45000|2023-08-10|  West|          40500.0|
|       5|        105|Desktop| 40000|2023-04-20| North|          36000.0|
|       3|        103| Tablet| 20000|2023-03-05|  West|          18000.0|
|       8|        103| Tablet| 20000|2023-07-05| North|          18000.0|
|       2|        102| Mobile| 15000|2023-02-10| South|          13500.0|
|       6|        101| Mobile| 15000|2023-05-10| South|          13500.0|
+--------+-----------+-------+------+-

In [158]:
# Sort the customer DataFrame by age in ascending order.
customer_df.sort(col('age').asc()).show()

+-----------+-------------+--------------------+---+-------------+---------------------+
|customer_id|customer_name|               email|age|customer_city|customer_age_category|
+-----------+-------------+--------------------+---+-------------+---------------------+
|        104|  Priya Patel|priya.patel@email...| 27|    Ahmedabad|                Youth|
|        101|  Arun Sharma|arun.sharma@email...| 28|        Delhi|                Youth|
|        105|  Sneha Reddy|sneha.reddy@email...| 29|    Hyderabad|                Youth|
|        103|  Rahul Yadav|rahul.yadav@email...| 30|    Bangalore|                Adult|
|        106|   Vikas Jain|vikas.jain@email.com| 31|      Chennai|                Adult|
|        102|  Meena Verma|meena.verma@email...| 34|       Mumbai|                Adult|
|        107|     Amit Roy|  amit.roy@email.com| 35|      Kolkata|                Adult|
+-----------+-------------+--------------------+---+-------------+---------------------+



# 8. Union Operations
* Add a new dataset for customers and perform a union operation with the customer
DataFrame.
* Combine the sales DataFrame with another DataFrame containing additional sales
records.

In [159]:
sales_new_df = spark.read.csv('sales_new.txt', header=True, inferSchema=True)
customer_new_df = spark.read.csv('customer_new.txt', header=True, inferSchema=True)

sales_new_df.show()
customer_new_df.show()

+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+-----------+-------+------+----------+------+
|      11|       4296| Laptop| 10299|2025-04-08| South|
|      12|       6307| Laptop| 16842|2025-11-25|  East|
|      13|       5369|Desktop| 64411|2025-04-13|  East|
|      14|       6125| Mobile| 55543|2024-12-15| North|
|      15|       9800| Laptop| 48545|2024-05-11|  East|
|      16|       5535| Laptop| 73010|2025-10-06| North|
|      17|       8351| Laptop| 15313|2024-07-13| North|
|      18|        247| Laptop| 50976|2025-05-20| South|
|      19|       4285| Laptop| 98067|2024-06-01|  West|
|      20|       2987| Laptop| 11911|2025-04-08| North|
+--------+-----------+-------+------+----------+------+

+-----------+------------------+--------------------+---+-----------------+
|customer_id|     customer_name|               email|age|             city|
+-----------+------------------+--------------------+---+------

In [160]:
# adding discount column for matching number of columns
sales_new_df = sales_new_df.withColumn('discounted_amount', round(col('amount') - col('amount') * 0.1, 2))
sales_new_df.show()

customer_new_df = customer_new_df.withColumn('customer_age_category',
                         when(col('age') < 30, 'Youth')
                         .when((col('age') >= 30) & (col('age') < 50), 'Adult')
                         .otherwise('Senior')
)

customer_new_df.show()

+--------+-----------+-------+------+----------+------+-----------------+
|sales_id|customer_id|product|amount| sale_date|region|discounted_amount|
+--------+-----------+-------+------+----------+------+-----------------+
|      11|       4296| Laptop| 10299|2025-04-08| South|           9269.1|
|      12|       6307| Laptop| 16842|2025-11-25|  East|          15157.8|
|      13|       5369|Desktop| 64411|2025-04-13|  East|          57969.9|
|      14|       6125| Mobile| 55543|2024-12-15| North|          49988.7|
|      15|       9800| Laptop| 48545|2024-05-11|  East|          43690.5|
|      16|       5535| Laptop| 73010|2025-10-06| North|          65709.0|
|      17|       8351| Laptop| 15313|2024-07-13| North|          13781.7|
|      18|        247| Laptop| 50976|2025-05-20| South|          45878.4|
|      19|       4285| Laptop| 98067|2024-06-01|  West|          88260.3|
|      20|       2987| Laptop| 11911|2025-04-08| North|          10719.9|
+--------+-----------+-------+------+-

In [161]:
# Add a new dataset for customers and perform a union operation with the customer DataFrame.
sales_df.union(sales_new_df).show()

+--------+-----------+-------+------+----------+------+-----------------+
|sales_id|customer_id|product|amount| sale_date|region|discounted_amount|
+--------+-----------+-------+------+----------+------+-----------------+
|       1|        101| Laptop| 50000|2023-01-15| North|          45000.0|
|       3|        103| Tablet| 20000|2023-03-05|  West|          18000.0|
|       7|        102| Laptop| 60000|2023-06-15|  East|          54000.0|
|      10|        105| Laptop| 70000|2023-09-25| North|          63000.0|
|       5|        105|Desktop| 40000|2023-04-20| North|          36000.0|
|       2|        102| Mobile| 15000|2023-02-10| South|          13500.0|
|       9|        104|Desktop| 45000|2023-08-10|  West|          40500.0|
|       6|        101| Mobile| 15000|2023-05-10| South|          13500.0|
|       8|        103| Tablet| 20000|2023-07-05| North|          18000.0|
|       4|        104| Laptop| 55000|2023-03-15|  East|          49500.0|
|      11|       4296| Laptop| 10299|2

In [162]:
# Combine the sales DataFrame with another DataFrame containing additional sales records.
customer_df.union(customer_new_df).show()

+-----------+------------------+--------------------+---+-----------------+---------------------+
|customer_id|     customer_name|               email|age|    customer_city|customer_age_category|
+-----------+------------------+--------------------+---+-----------------+---------------------+
|        101|       Arun Sharma|arun.sharma@email...| 28|            Delhi|                Youth|
|        102|       Meena Verma|meena.verma@email...| 34|           Mumbai|                Adult|
|        103|       Rahul Yadav|rahul.yadav@email...| 30|        Bangalore|                Adult|
|        104|       Priya Patel|priya.patel@email...| 27|        Ahmedabad|                Youth|
|        105|       Sneha Reddy|sneha.reddy@email...| 29|        Hyderabad|                Youth|
|        106|        Vikas Jain|vikas.jain@email.com| 31|          Chennai|                Adult|
|        107|          Amit Roy|  amit.roy@email.com| 35|          Kolkata|                Adult|
|        111|Melissa

# 9. Window Functions
* Rank the sales records based on the amount column.
* Add a cumulative sum of amount for each product in the sales DataFrame.
* Add a column that calculates the difference between each customer's amount and the
average amount within their product group.

In [163]:
# Rank the sales records based on the amount column.
window_spec_rank = Window.orderBy(col('amount').desc())
sales_df.withColumn('rank', rank().over(window_spec_rank)).show()

+--------+-----------+-------+------+----------+------+-----------------+----+
|sales_id|customer_id|product|amount| sale_date|region|discounted_amount|rank|
+--------+-----------+-------+------+----------+------+-----------------+----+
|      10|        105| Laptop| 70000|2023-09-25| North|          63000.0|   1|
|       7|        102| Laptop| 60000|2023-06-15|  East|          54000.0|   2|
|       4|        104| Laptop| 55000|2023-03-15|  East|          49500.0|   3|
|       1|        101| Laptop| 50000|2023-01-15| North|          45000.0|   4|
|       9|        104|Desktop| 45000|2023-08-10|  West|          40500.0|   5|
|       5|        105|Desktop| 40000|2023-04-20| North|          36000.0|   6|
|       3|        103| Tablet| 20000|2023-03-05|  West|          18000.0|   7|
|       8|        103| Tablet| 20000|2023-07-05| North|          18000.0|   7|
|       2|        102| Mobile| 15000|2023-02-10| South|          13500.0|   9|
|       6|        101| Mobile| 15000|2023-05-10| Sou

In [164]:
# Add a cumulative sum of amount for each product in the sales DataFrame.
window_spec_1 = Window.partitionBy('product')\
                      .orderBy('sale_date')\
                      .rowsBetween(Window.unboundedPreceding, Window.currentRow)

sales_df.withColumn('cummulative_sum', sum('amount').over(window_spec_1)).show()

+--------+-----------+-------+------+----------+------+-----------------+---------------+
|sales_id|customer_id|product|amount| sale_date|region|discounted_amount|cummulative_sum|
+--------+-----------+-------+------+----------+------+-----------------+---------------+
|       5|        105|Desktop| 40000|2023-04-20| North|          36000.0|          40000|
|       9|        104|Desktop| 45000|2023-08-10|  West|          40500.0|          85000|
|       1|        101| Laptop| 50000|2023-01-15| North|          45000.0|          50000|
|       4|        104| Laptop| 55000|2023-03-15|  East|          49500.0|         105000|
|       7|        102| Laptop| 60000|2023-06-15|  East|          54000.0|         165000|
|      10|        105| Laptop| 70000|2023-09-25| North|          63000.0|         235000|
|       2|        102| Mobile| 15000|2023-02-10| South|          13500.0|          15000|
|       6|        101| Mobile| 15000|2023-05-10| South|          13500.0|          30000|
|       3|

In [165]:
''' Add a column that calculates the difference between each customer's amount and
the average amount within their product group. '''
window_spec_avg = Window.partitionBy('product')\
                      .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

sales_df = sales_df.withColumn('avg', avg('amount').over(window_spec_avg))\
                   .withColumn('diff_avg_amount', col('amount') - col('avg'))

sales_df.show()

+--------+-----------+-------+------+----------+------+-----------------+-------+---------------+
|sales_id|customer_id|product|amount| sale_date|region|discounted_amount|    avg|diff_avg_amount|
+--------+-----------+-------+------+----------+------+-----------------+-------+---------------+
|       5|        105|Desktop| 40000|2023-04-20| North|          36000.0|42500.0|        -2500.0|
|       9|        104|Desktop| 45000|2023-08-10|  West|          40500.0|42500.0|         2500.0|
|       1|        101| Laptop| 50000|2023-01-15| North|          45000.0|58750.0|        -8750.0|
|       7|        102| Laptop| 60000|2023-06-15|  East|          54000.0|58750.0|         1250.0|
|      10|        105| Laptop| 70000|2023-09-25| North|          63000.0|58750.0|        11250.0|
|       4|        104| Laptop| 55000|2023-03-15|  East|          49500.0|58750.0|        -3750.0|
|       2|        102| Mobile| 15000|2023-02-10| South|          13500.0|15000.0|            0.0|
|       6|        10

**10. Partitioning**
* Write the sales DataFrame to a partitioned Parquet file by region.
* Partition the customer DataFrame by customer_city and save it as a CSV file.

In [166]:
# Write the sales DataFrame to a partitioned Parquet file by region.
sales_df.write.partitionBy('region').parquet('sales_cleaned_data_regionwise', mode='overwrite')

In [167]:
# Partition the customer DataFrame by customer_city and save it as a CSV file.
customer_df.write.partitionBy('customer_city').csv('customer_cleaned_data_citywise', header=True, mode='overwrite')

**11. Real-World Scenarios**
* Calculate the percentage contribution of each product to the total sales.
* Extract the year from sale_date and group by year to calculate total sales.
* Identify the most purchased product in each region.
* Add a column to show the difference between the highest and lowest sales for each product.
* Write the result of the join between sales and customer to parquet file.
* Identify products that were sold in the last 6 months.
* Calculate the average sales amount per customer.

In [168]:
# Calculate the percentage contribution of each product to the total sales.
total_sales = sales_df.groupBy().sum('amount').collect()[0][0]
print('total_sales :- ', total_sales)

sales_df.groupBy('product')\
        .agg(sum('amount').alias('sales_product_wise'))\
        .withColumn('percent_contribution', round(col('sales_product_wise') / total_sales * 100, 2))\
        .show()

total_sales :-  390000
+-------+------------------+--------------------+
|product|sales_product_wise|percent_contribution|
+-------+------------------+--------------------+
| Laptop|            235000|               60.26|
| Mobile|             30000|                7.69|
| Tablet|             40000|               10.26|
|Desktop|             85000|               21.79|
+-------+------------------+--------------------+



In [169]:
# Extract the year from sale_date and group by year to calculate total sales.
sales_df.withColumn('year', year('sale_date'))\
        .groupBy('year')\
        .agg(sum('amount').alias('total_sales'))\
        .show()

+----+-----------+
|year|total_sales|
+----+-----------+
|2023|     390000|
+----+-----------+



In [170]:
# Identify the most purchased product in each region.
window_spec_dense_rank = Window.partitionBy('region').orderBy(col('product_count').desc())

sales_df.groupBy('region', 'product')\
        .agg(count('product').alias('product_count'))\
        .withColumn('dense_rank', dense_rank().over(window_spec_dense_rank))\
        .where('dense_rank == 1')\
        .show()

+------+-------+-------------+----------+
|region|product|product_count|dense_rank|
+------+-------+-------------+----------+
|  East| Laptop|            2|         1|
| North| Laptop|            2|         1|
| South| Mobile|            2|         1|
|  West|Desktop|            1|         1|
|  West| Tablet|            1|         1|
+------+-------+-------------+----------+



In [171]:
# Add a column to show the difference between the highest and lowest sales for each product.
sales_df.groupBy('product')\
        .agg(max('amount').alias('highest_sales'), min('amount').alias('lowest_sales'))\
        .withColumn('difference_between', col('highest_sales') - col('lowest_sales'))\
        .show()

+-------+-------------+------------+------------------+
|product|highest_sales|lowest_sales|difference_between|
+-------+-------------+------------+------------------+
| Laptop|        70000|       50000|             20000|
| Mobile|        15000|       15000|                 0|
| Tablet|        20000|       20000|                 0|
|Desktop|        45000|       40000|              5000|
+-------+-------------+------------+------------------+



In [172]:
# Write the result of the join between sales and customer to parquet file.
sales_df.join(customer_df, on = 'customer_id', how = 'inner').write.parquet('joined_data_sales_customer', mode='overwrite')


In [173]:
# Identify products that were sold in the last 6 months.
sales_df.select('product', 'sale_date')\
        .where(col('sale_date') >= add_months(current_date(), -6))\
        .groupBy('product').agg(count('product').alias('count'))\
        .show()

# there are no products sold in last 6 months thus no outputs thus printing last 60 months or 5 years
sales_df.select('product', 'sale_date')\
        .where(col('sale_date') >= add_months(current_date(), -60))\
        .groupBy('product').agg(count('product').alias('count'))\
        .show()

+-------+-----+
|product|count|
+-------+-----+
+-------+-----+

+-------+-----+
|product|count|
+-------+-----+
| Laptop|    4|
| Mobile|    2|
| Tablet|    2|
|Desktop|    2|
+-------+-----+



In [174]:
# Calculate the average sales amount per customer.
sales_df.groupBy('customer_id')\
        .agg(avg('amount').alias('avg_sales_per_customer'))\
        .orderBy('customer_id')\
        .show()

+-----------+----------------------+
|customer_id|avg_sales_per_customer|
+-----------+----------------------+
|        101|               32500.0|
|        102|               37500.0|
|        103|               20000.0|
|        104|               50000.0|
|        105|               55000.0|
+-----------+----------------------+

